# 🔬 Notebook 10: Ablation Study (Extension 2)

**Objective:** Step-wise ablation analysis to quantify the incremental contribution of each feature category.

Features are added in this order:
1. Base pollutants only
2. + Lag features
3. + Rolling average features
4. + Cyclic temporal features
5. + Ratio features
6. + Meteorological features

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data.data_loader import load_config, load_csv
from src.preprocessing.scaler import AQIScaler
from src.models.cnn_lstm import build_cnn_lstm_model
from src.evaluation.ablation_analysis import run_ablation_study, plot_ablation_results

config = load_config('../config/config.yaml')

# Load featured data
df = load_csv('../' + config['data']['featured_path'])
print(f"Shape: {df.shape}")

# Load feature groups
with open('../artifacts/feature_groups.json', 'r') as f:
    feature_groups = json.load(f)

print("Feature groups:")
for name, cols in feature_groups.items():
    print(f"  {name}: {len(cols)} features")

## 10.1 Scale Data for Ablation

In [ ]:
# Get all feature columns
exclude_cols = ['aqi', 'aqi_category', 'aqi_code']
all_feature_cols = [c for c in df.columns if c not in exclude_cols]

# Scale using training portion only
split_idx = int(len(df) * 0.70)
scaler = AQIScaler()
scaler.fit(df.iloc[:split_idx], all_feature_cols)
df_scaled = scaler.transform(df)

print(f"Scaled data shape: {df_scaled.shape}")

## 10.2 Run Step-wise Ablation Study

In [ ]:
# Run ablation using the Hybrid CNN-LSTM as the test model
ablation_results = run_ablation_study(
    df=df_scaled,
    feature_groups=feature_groups,
    target_col='aqi_code',
    model_builder=lambda shape, nc: build_cnn_lstm_model(shape, nc, config['models']['cnn_lstm']),
    train_config=config['training'],
    window_size=config['sequence']['window_size'],
    test_size=config['split']['test_size'],
    results_dir='../results'
)

ablation_results

## 10.3 Visualize Ablation Results

In [ ]:
plot_ablation_results(ablation_results,
                      save_path='../results/ablation_study.png')

In [ ]:
# Summary table
print("\nAblation Study Summary:")
print("=" * 70)
for _, row in ablation_results.iterrows():
    delta_f1 = ''
    if row['step'] > 1:
        prev_f1 = ablation_results.iloc[int(row['step'])-2]['f1_score']
        diff = row['f1_score'] - prev_f1
        delta_f1 = f"  (Δ = {diff:+.4f})"
    print(f"  Step {int(row['step'])}: +{row['feature_group']:20s} → "
          f"F1 = {row['f1_score']:.4f}{delta_f1}")
print("=" * 70)

## ✅ Ablation Study Complete

**Next:** Notebook 11 — Meteorological Extension